# 03. Map-Refine 요약

Map-Reduce 와 비슷하지만, 합치는 방식이 다릅니다.

1. **Map**: 각 chunk 를 개별적으로 요약합니다. (병렬)
2. **Refine**: 첫 번째 요약에서 시작해, 다음 요약을 하나씩 반영하며 **최종 요약을 점진적으로 다듬습니다.** (순차)

- 장점: 문서의 **순서와 맥락**을 유지하면서 요약할 수 있습니다.
- 단점: Refine 단계는 이전 결과가 있어야 다음을 진행할 수 있어 **병렬화가 불가능** → 문서가 많으면 느립니다.

## 환경 설정

- API 키(`OPENAI_API_KEY`, `LANGSMITH_API_KEY` 등)는 `.env` 파일 또는 시스템 환경변수에서 읽어옵니다.
- `logging.langsmith()` 를 호출하면 실행 과정이 LangSmith 에 기록되어 체인 내부 동작을 확인할 수 있습니다.

In [1]:
# .env 파일에 저장된 API KEY 를 환경변수로 불러옵니다.
from dotenv import load_dotenv

load_dotenv()

# LangSmith 추적 설정 (프로젝트 이름: Summary)
from langchain_teddynote import logging

logging.langsmith("Summary")

LangSmith 추적을 시작합니다.
[프로젝트명]
Summary


## 데이터 로드

**실습 문서**: 소프트웨어정책연구소(SPRi) AI Brief 2023년 12월호 (`data/SPRI_AI_Brief_2023년12월호_F.pdf`)

In [2]:
from langchain_community.document_loaders import PyPDFLoader

# PDF 를 페이지 단위로 로드합니다. (1페이지 = Document 1개)
loader = PyPDFLoader("../data/SPRI_AI_Brief_2023년12월호_F.pdf")
docs = loader.load()
print(f"전체 페이지 수: {len(docs)}")

# 실습 비용/시간을 줄이기 위해 일부 페이지(4~8쪽)만 요약합니다.
docs = docs[3:8]
print(f"요약할 페이지 수: {len(docs)}")

C:\Users\user\AppData\Local\Temp\ipykernel_12104\1564795920.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


전체 페이지 수: 23
요약할 페이지 수: 5


### 프롬프트 가져오기 (LangSmith Hub)

> ⚠️ **레거시 변경점**
> 예전 코드의 `from langchain import hub` / `hub.pull(...)` 은 LangChain 1.x 에서 `langchain_classic` 으로 밀려났고,
> 최신 `langsmith` 는 **다른 사람(owner)의 공개 프롬프트**를 기본적으로 막습니다.
> 그래서 `langsmith.Client().pull_prompt(..., dangerously_pull_public_prompt=True)` 로 직접 가져옵니다.
> (`teddynote` 튜토리얼 프롬프트처럼 **신뢰할 수 있는 프롬프트에만** 사용하세요.)

In [3]:
from langsmith import Client

client = Client()


def pull_prompt(name: str):
    """LangSmith Hub 에서 공개 프롬프트를 가져옵니다. (hub.pull 대체)"""
    return client.pull_prompt(name, dangerously_pull_public_prompt=True)

## Map 단계

각 페이지를 지정한 언어(`language`)로 bullet 요약합니다.

In [4]:
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# map 요약 프롬프트 (입력 변수: documents, language)
map_summary_prompt = pull_prompt("teddynote/map-summary-prompt")
map_summary_prompt.pretty_print()

================================ System Message ================================

You are an expert summarizer. Your task is to summarize the following document in {language}.

================================ Human Message =================================

Extract most important main thesis from the documents, then summarize in bullet points.

#Format:
- summary 1
- summary 2
- summary 3
-...

Here is a given document: 
{documents}

Write 1~5 sentences. Think step by step.
#Summary:


In [5]:
map_chain = map_summary_prompt | llm | StrOutputParser()

# 첫 번째 페이지만 먼저 요약해 봅니다.
print(map_chain.invoke({"documents": docs[0].page_content, "language": "Korean"}))

- 미국 바이든 대통령이 안전하고 신뢰할 수 있는 AI 개발과 사용을 위한 행정명령을 발표하였다.
- 이 행정명령은 AI의 안전과 보안 기준, 개인정보 보호, 형평성과 시민권 향상, 소비자 보호, 노동자 지원, 혁신과 경쟁 촉진, 국제 협력을 포함한다.
- AI 시스템 개발 기업은 안전 테스트 결과와 시스템 정보를 정부와 공유해야 하며, AI의 책임 있는 사용을 위한 모범사례와 지침이 마련된다.
- 소비자 보호와 근로자 지원을 위해 의료 분야에서의 AI 사용 촉진과 교육 도구 개발이 강조된다.
- 국가 AI 연구 자원을 통해 AI 연구를 촉진하고, 외국인 전문가들이 미국에서 공부하고 일할 수 있도록 지원하는 방안도 포함된다.


In [6]:
# 모든 페이지를 입력 형식(dict)으로 변환한 뒤 병렬 요약합니다.
input_docs = [{"documents": doc.page_content, "language": "Korean"} for doc in docs]
doc_summaries = map_chain.batch(input_docs)

for i, s in enumerate(doc_summaries, start=1):
    print(f"===== 페이지 {i} =====\n{s}\n")

===== 페이지 1 =====
- 미국 바이든 대통령이 안전하고 신뢰할 수 있는 AI 개발과 사용을 위한 행정명령을 발표하였다.
- 이 행정명령은 AI의 안전과 보안 기준, 개인정보 보호, 형평성과 시민권 향상, 소비자 보호, 노동자 지원, 혁신과 경쟁 촉진, 국제 협력을 포함한다.
- AI 시스템 개발 기업은 안전 테스트 결과와 시스템 정보를 정부와 공유해야 하며, AI의 책임 있는 사용을 위한 모범사례와 지침이 마련된다.
- 소비자 보호와 근로자 지원을 위해 의료 분야에서의 AI 사용 촉진 및 교육 도구 개발이 강조된다.
- 국가 AI 연구 자원을 통해 AI 연구를 촉진하고, 외국인 전문가들이 미국에서 공부하고 일할 수 있도록 지원하는 방안도 포함된다.

===== 페이지 2 =====
- G7은 2023년 10월 30일 '히로시마 AI 프로세스'를 통해 AI 기업을 위한 국제 행동강령에 합의하였다.
- 이 행동강령은 AI 시스템의 위험 식별 및 완화를 위한 자발적 채택을 권장하며, AI 수명주기 전반에 걸친 위험 평가와 투명성, 책임성을 강조한다.
- 주요 내용으로는 정보 공유, 보안 통제, 콘텐츠 인증 및 출처 확인 메커니즘 개발 등이 포함된다.
- G7은 AI 기술의 빠른 발전에 대응하기 위해 이해관계자 간 협의를 통해 행동강령을 지속적으로 개정할 예정이다.
- 또한, 사회적 위험 완화와 글로벌 문제 해결을 위한 AI 시스템 개발에 우선 투자할 계획이다.

===== 페이지 3 =====
- 28개국이 영국 블레츨리 파크에서 열린 AI 안전성 정상회의에서 AI 안전 보장을 위한 블레츨리 선언을 발표했다.
- 선언은 AI 시스템의 안전성을 보장하기 위해 모든 이해관계자의 협력이 필요하다고 강조하며, 특히 AI 개발 기업의 책임을 지적했다.
- 영국 총리는 AI 안전 연구소의 출범과 함께 첨단 AI 모델에 대한 안전성 시험 계획을 발표하고, 각국 정부는 테스트 결과를 공유하고 공동 표준 개발을 위해 협력하기로 합의했다.
- 참가국들은 AI 위험과 가능성

## Refine 단계

이전 요약(`previous_summary`)에 새 요약(`current_summary`)을 반영해 요약을 갱신합니다.

In [7]:
# refine 프롬프트 (입력 변수: previous_summary, current_summary, language)
refine_prompt = pull_prompt("teddynote/refine-prompt")
refine_prompt.pretty_print()

================================ System Message ================================

You are an expert summarizer.

================================ Human Message =================================

Your job is to produce a final summary

We have provided an existing summary up to a certain point:
{previous_summary}

We have the opportunity to refine the existing summary(only if needed) with some more context below.
------------
{current_summary}
------------
Given the new context, refine the original summary in {language}.
If the context isn't useful, return the original summary.


In [8]:
refine_chain = refine_prompt | llm | StrOutputParser()

# 첫 요약을 시작점으로, 나머지 요약을 순서대로 반영합니다.
previous_summary = doc_summaries[0]
for current_summary in doc_summaries[1:]:
    previous_summary = refine_chain.invoke(
        {
            "previous_summary": previous_summary,
            "current_summary": current_summary,
            "language": "Korean",
        }
    )

print(previous_summary)

- 미국 바이든 대통령이 안전하고 신뢰할 수 있는 AI 개발과 사용을 위한 행정명령을 발표하였다.
- 이 행정명령은 AI의 안전과 보안 기준, 개인정보 보호, 형평성과 시민권 향상, 소비자 보호, 노동자 지원, 혁신과 경쟁 촉진, 국제 협력을 포함한다.
- AI 시스템 개발 기업은 안전 테스트 결과와 시스템 정보를 정부와 공유해야 하며, AI의 책임 있는 사용을 위한 모범사례와 지침이 마련된다.
- 소비자 보호와 근로자 지원을 위해 의료 분야에서의 AI 사용 촉진 및 교육 도구 개발이 강조된다.
- 국가 AI 연구 자원을 통해 AI 연구를 촉진하고, 외국인 전문가들이 미국에서 공부하고 일할 수 있도록 지원하는 방안도 포함된다.
- G7은 '히로시마 AI 프로세스'를 통해 AI 기업을 위한 국제 행동강령에 합의하였으며, 이는 AI 시스템의 위험 식별 및 완화를 위한 자발적 채택을 권장하고, AI 수명주기 전반에 걸친 위험 평가와 투명성, 책임성을 강조한다.
- G7은 AI 기술의 빠른 발전에 대응하기 위해 이해관계자 간 협의를 통해 행동강령을 지속적으로 개정할 계획이며, 사회적 위험 완화와 글로벌 문제 해결을 위한 AI 시스템 개발에 우선 투자할 예정이다.
- 또한, 28개국이 영국 블레츨리 파크에서 열린 AI 안전성 정상회의에서 AI 안전 보장을 위한 블레츨리 선언을 발표하였다. 이 선언은 AI 시스템의 안전성을 보장하기 위해 모든 이해관계자의 협력이 필요하다고 강조하며, 특히 AI 개발 기업의 책임을 지적하였다.
- 영국 총리는 AI 안전 연구소의 출범과 함께 첨단 AI 모델에 대한 안전성 시험 계획을 발표하고, 각국 정부는 테스트 결과를 공유하고 공동 표준 개발을 위해 협력하기로 합의하였다.
- 참가국들은 AI 위험과 가능성에 대한 과학적 평가를 위한 보고서 작성을 합의하였으며, 한국은 영국과 AI 미니 정상회의를 공동 개최할 예정이다.
- 한편, 미국 캘리포니아 북부지방법원은 예술가들이 미드저니, 스태빌리티AI, 디비언트아트를 상대로 제기한 저작권 침해 

## Map-Refine 을 하나의 체인으로 묶기

05(Clustering) 노트북에서도 같은 함수를 재사용합니다.

In [9]:
from langchain_core.runnables import chain


@chain
def map_refine_chain(docs):
    """Document 리스트를 받아 Map-Refine 방식으로 요약합니다."""
    # 1) Map: 각 문서를 병렬로 요약
    map_chain = pull_prompt("teddynote/map-summary-prompt") | llm | StrOutputParser()
    doc_summaries = map_chain.batch(
        [{"documents": doc.page_content, "language": "Korean"} for doc in docs]
    )

    # 2) Refine: 첫 요약에서 시작해 나머지 요약을 하나씩 반영하며 다듬기 (순차 실행)
    refine_chain = pull_prompt("teddynote/refine-prompt") | llm | StrOutputParser()
    previous_summary = doc_summaries[0]
    for i, current_summary in enumerate(doc_summaries[1:], start=2):
        previous_summary = refine_chain.invoke(
            {
                "previous_summary": previous_summary,
                "current_summary": current_summary,
                "language": "Korean",
            }
        )
        print(f"[refine {i}/{len(doc_summaries)}] 완료")

    return previous_summary

In [10]:
refined_summary = map_refine_chain.invoke(docs)
print("\n============== [최종 요약] ==============\n")
print(refined_summary)

[refine 2/5] 완료
[refine 3/5] 완료
[refine 4/5] 완료
[refine 5/5] 완료

============== [최종 요약] ==============

- 미국 바이든 대통령이 안전하고 신뢰할 수 있는 AI 개발과 사용을 위한 행정명령을 발표하였다.
- 이 행정명령은 AI의 안전과 보안 기준, 개인정보 보호, 형평성과 시민권 향상, 소비자 보호, 노동자 지원, 혁신과 경쟁 촉진, 국제 협력을 포함한다.
- AI 시스템 개발 기업은 안전 테스트 결과와 시스템 정보를 정부와 공유해야 하며, AI의 책임 있는 사용을 위한 모범사례와 지침이 마련된다.
- 소비자 보호와 근로자 지원을 위해 의료 분야에서의 AI 사용 촉진 및 교육 도구 개발이 강조된다.
- 국가 AI 연구 자원을 통해 AI 연구를 촉진하고, 외국인 전문가들이 미국에서 공부하고 일할 수 있도록 지원하는 방안도 포함된다.
- G7은 '히로시마 AI 프로세스'를 통해 AI 기업을 위한 국제 행동강령에 합의하였으며, 이는 AI 시스템의 위험 식별 및 완화를 위한 자발적 채택을 권장하고, 위험 평가와 투명성, 책임성을 강조한다.
- G7은 정보공유 및 협력, 보안 통제, 콘텐츠 인증 및 출처 확인 메커니즘 개발 등을 포함한 행동강령을 지속적으로 개정할 계획이다.
- 또한, 28개국이 영국 블레츨리 파크에서 열린 AI 안전성 정상회의에서 AI 안전 보장을 위한 블레츨리 선언을 발표하였으며, AI 시스템의 안전성을 보장하기 위해 모든 이해관계자의 협력이 중요하다고 강조하였다.
- 영국 총리는 AI 안전 연구소의 출범과 함께 첨단 AI 모델에 대한 안전성 시험 계획을 발표하고, 외부 안전 테스트에 대한 합의를 이끌어냈다.
- 각국은 AI 안전 보장을 위해 투명성 향상, 안전 테스트 도구 개발, 공공부문 역량 구축 등에서 협력하기로 합의하였으며, 한국은 영국과 AI 미니 정상회의를 공동 개최하고, 프랑스와는 대면 정상회의를 계획하고 있다.
- 한편, 미국 캘리포니아 북부지방법원은 예술